In [9]:
import pandas as pd
import numpy as np
import random

# 1. Generate file

In [18]:
df = pd.read_csv('manipulasi.csv', sep=';')

# list kelas
kelas_c = ['C'] * 43
kelas_d = ['D'] * 42
kolom_kelas = kelas_c + kelas_d

random.seed(42)
random.shuffle(kolom_kelas)

if len(df) == len(kolom_kelas):
    df['kelas'] = kolom_kelas
else:
    df['kelas'] = kolom_kelas[:len(df)] 

df['cukup atau tidak'] = df['cukup atau tidak'].astype(str).str.strip().str.lower()
df['is_cukup'] = df['cukup atau tidak'].apply(lambda x: 1 if x == 'ya' else 0)

# save file
df.to_csv('data_final.csv', sep=';', index=False)
print("Selesai\n")

Selesai



# 2 Analisis

## 2.1 Cluster samplling

In [10]:
# parameter
N = 5  # jumlah kelas populasi
n = df['kelas'].nunique()  # jumlah kelas sampel
f = n / N  # sampling fraction

# agregat per klaster
cluster_stats = df.groupby('kelas').agg(
    y_i=('waktu belajar', 'sum'),
    a_i=('is_cukup', 'sum'),
    m_i=('kelas', 'count')
).reset_index()

m_bar = cluster_stats['m_i'].mean() # rata-rata ukuran sampel per klaster

## 2.2 Estimasi rata-rata waktu belajar

In [15]:
# estimasi rasio untuk Rata-rata
y_bar_est = cluster_stats['y_i'].sum() / cluster_stats['m_i'].sum()

# varians dan standard error untuk rata-rata
cluster_stats['sq_diff_mean'] = (cluster_stats['y_i'] - y_bar_est * cluster_stats['m_i'])**2
var_y_bar = ((1 - f) / (n * (m_bar**2))) * (cluster_stats['sq_diff_mean'].sum() / (n - 1))
se_y_bar = np.sqrt(var_y_bar)

# margin error, CI = 95%
moe_y_bar = 1.96 * se_y_bar

print("\nEstimasi Rata-rata Waktu Belajar:")
print(f"Estimasi titik (rata-rata): {y_bar_est:.2f} menit/hari")
print(f"Standard error (SE): {se_y_bar:.2f}")
print(f"Interval kepercayaan (95%): {y_bar_est - moe_y_bar:.2f} < \u03bc < {y_bar_est + moe_y_bar:.2f} menit")



Estimasi Rata-rata Waktu Belajar:
Estimasi titik (rata-rata): 112.24 menit/hari
Standard error (SE): 2.90
Interval kepercayaan (95%): 106.56 < μ < 117.91 menit


## 2.3 Estimasi kecukupan waktu

In [21]:
# estimasi ratio untuk proporsi
p_est = cluster_stats['a_i'].sum() / cluster_stats['m_i'].sum()

# varians dan standard rrror untuk proporsi
cluster_stats['sq_diff_prop'] = (cluster_stats['a_i'] - p_est * cluster_stats['m_i'])**2
var_p = ((1 - f) / (n * (m_bar**2))) * (cluster_stats['sq_diff_prop'].sum() / (n - 1))
se_p = np.sqrt(var_p)

# margin of error, CI 95%
moe_p = 1.96 * se_p

print("\nEstimasi Kecukupan Waktu:")
print(f"Estimasi titik (proporsi): {p_est:.2%} ({p_est:.4f})")
print(f"Standard error (SE): {se_p:.4f}")
print(f"Interval kepercayaan (95%): {(p_est - moe_p)*100:.2f}% < \u03c1 < {(p_est + moe_p)*100:.2f}%")


Estimasi Kecukupan Waktu:
Estimasi titik (proporsi): 3.53% (0.0353)
Standard error (SE): 0.0094
Interval kepercayaan (95%): 1.68% < ρ < 5.38%
